# 02 · Data cleaning results
**Project:** Sales Variance FP&A Dashboard

The cleaning is done by `scripts/clean_sales_data.py`. It reads `data/raw/sales_raw_messy.xlsx` and writes
`data/processed/Cleaned_Sales_Data.csv` plus `Cleaned_Sales_Data.xlsx` (the same data with a full audit trail).

This notebook **checks the result**. It does not clean anything itself.

Rules the cleaner follows:
* Every changed value is listed with its original value and the rule used.
* A number is repaired only when `Sales = Qty x Price` and `Gross_Profit = Sales - Qty x Cost` give the **exact** answer. Nothing is estimated; anything uncertain goes to review.
* Labels are mapped with explicit tables (no fuzzy matching).

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Works whether Jupyter starts in notebooks/ or in the repository root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from fpa_lib import config  # noqa: E402

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_columns = 40
pd.options.display.max_colwidth = 90
rel = lambda p: p.relative_to(ROOT).as_posix()  # noqa: E731  (print paths without your local folder names)
import importlib.util  # noqa: E402

clean = pd.read_csv(config.CLEAN_CSV, dtype={"Line_ID": str, "Notes": str}, keep_default_na=False, na_values=[""])
book = pd.read_excel(config.CLEAN_XLSX, sheet_name=None)
print(f"Clean file: {rel(config.CLEAN_CSV)}  rows: {len(clean):,}")
print("Workbook sheets:", ", ".join(book))

Clean file: data/processed/Cleaned_Sales_Data.csv  rows: 20,900
Workbook sheets: Read_Me, Clean_Data, Row_Reconciliation, Cleaning_Log, Corrections_Audit, Review_Required, Removed_Duplicates, Value_Mappings, Data_Quality, Validation_Checks


## 1. Row reconciliation
Every raw row is accounted for.

In [2]:
recon = book["Row_Reconciliation"]
assert recon["Rows"].iloc[:-1].sum() == recon["Rows"].iloc[-1] == len(clean)
recon

,Step,Rows
0,Raw data rows,21815
1,- entirely blank rows,-25
2,- exact duplicate rows,-525
3,- duplicate copies merged,-252
4,- rows held for review,-113
5,= clean rows,20900


## 2. What was done (cleaning log)

In [3]:
book["Cleaning_Log"]

,Step,Column,Issue,Rows_Affected,Action,Certainty
0,1. Schema,"Actual_Qty, Target_Sales_USD",Header had extra spaces,2,Trimmed header names,Standardised
1,1. Schema,Comments,Column is entirely empty,0,Dropped the column,Standardised
2,2. Rows,All,Entirely blank rows,25,Dropped,Standardised
3,2. Rows,All,Exact duplicate rows,525,Removed - first copy kept,Exact
4,3. Parsing,Text and number columns,Numbers stored as text,10418,Parsed / trimmed (value unchanged),Standardised
5,3. Parsing,Text and number columns,Hidden whitespace,4124,Parsed / trimmed (value unchanged),Standardised
6,3. Parsing,Text and number columns,Null spellings,2232,Parsed / trimmed (value unchanged),Standardised
7,3. Parsing,Text and number columns,Date text/serial formats,6162,Parsed / trimmed (value unchanged),Standardised
8,4. Key,Line_ID,Repeated Line_ID with consistent values,252,Merged into the first row (blanks filled from the copy),Exact
9,4. Key,Line_ID,Repeated Line_ID with conflicting values,20,All copies held for review (no winner chosen),Needs review


## 3. Repairs by column and rule
All repairs are `Exact`: solved by an identity, by the Line_ID key or by the product master.

In [4]:
audit = book["Corrections_Audit"]
print(f"Values repaired: {len(audit):,}   certainty: {audit['Certainty'].value_counts().to_dict()}")
audit.assign(Rule=audit["Rule"].str.replace(r"row \d+", "row N", regex=True)).groupby(["Column", "Rule"]).size()     .rename("values").to_frame()

Values repaired: 4,672   certainty: {'Exact': 4672}


values
Column                  Rule                                                                                              
Actual_Gross_Profit_USD Blank - filled from duplicate row N                                                              2
                        Missing - Gross_Profit = Sales - Qty x Unit_Cost                                               310
                        Wrong value - Gross_Profit = Sales - Qty x Unit_Cost (the only single change that makes...      84
Actual_Qty              Blank - filled from duplicate row N                                                              1
                        Missing - Qty = Sales / Unit_Price                                                             520
                        Wrong value - Qty = Sales / Unit_Price (the only single change that makes both identiti...     231
Actual_Sales_USD        Missing - Sales = Qty x Unit_Price                                                             205
Actual_Unit_Cost_USD    Blank - filled from duplicate row N                                                             42
                        Missing - Unit_Cost = (Sales - Gross_Profit) / Qty                                             312
Actual_Unit_Price_USD   Missing - Unit_Price = Sales / Qty                                                             313
                        Wrong value - Unit_Price = Sales / Qty (the only single change that makes both identiti...      42
Category                Blank - recovered from product master                                                          416
Date                    Blank - recovered from Line_ID                                                                 208
Month_Name              Recomputed from Date (derived field)                                                           209
Notes                   Blank - filled from duplicate row N                                                              6
Region                  Blank - recovered from Line_ID                                                                 619
Target_Gross_Profit_USD Blank - filled from duplicate row N                                                             48
                        Missing - Gross_Profit = Sales - Qty x Unit_Cost                                               209
Target_Qty              Blank - filled from duplicate row N                                                              3
                        Missing - Qty = Sales / Unit_Price                                                             201
Target_Sales_USD        Blank - filled from duplicate row N                                                              1
                        Missing - Sales = Qty x Unit_Price                                                             164
                        Wrong value - Sales = Qty x Unit_Price (the only single change that makes both identiti...      83
Target_Unit_Cost_USD    Blank - filled from duplicate row N                                                              1
                        Missing - Unit_Cost = (Sales - Gross_Profit) / Qty                                             200
Target_Unit_Price_USD   Blank - filled from duplicate row N                                                             41
                        Missing - Unit_Price = Sales / Qty                                                             201

## 4. Rows held for review
These are excluded from the clean data and listed with the reason. A person has to decide them.

In [5]:
review = book["Review_Required"]
reasons = review["Review_Reason"].str.replace(r"'[^']*'", "'…'", regex=True).str.replace(r"\(.*?\)", "(…)", regex=True)
print(f"Rows held for review: {len(review):,}")
reasons.value_counts().to_frame("rows")

Rows held for review: 113


,rows
Review_Reason,
"Line_ID repeated with conflicting values in Actual_Qty, Actual_Sales_USD, Actual_Gross_Profit_USD - all copies held for review",32
"Cannot recover Actual_Qty, Actual_Sales_USD exactly (…)",24
"Cannot recover Actual_Unit_Cost_USD, Actual_Gross_Profit_USD exactly (…)",15
Region '…' contradicts Line_ID (…),12
Product '…' contradicts Line_ID (…),8
Date '…' contradicts Line_ID month (…),8
Missing or malformed Line_ID (…),6
"Line_ID repeated with conflicting values in Actual_Sales_USD, Actual_Gross_Profit_USD - all copies held for review",2
"Line_ID repeated with conflicting values in Actual_Qty, Actual_Gross_Profit_USD - all copies held for review",2


## 5. Independent check against the ground truth
Because the data is synthetic, the true values are known. The generator is re-run in memory (same seed) and every cell of the clean file is compared with the truth.

In [6]:
spec = importlib.util.spec_from_file_location("gen", ROOT / "scripts" / "generate_synthetic_data.py")
gen = importlib.util.module_from_spec(spec)
spec.loader.exec_module(gen)
truth = gen.build_truth(config.RANDOM_SEED)

m = truth.merge(clean, on="Line_ID", suffixes=("_t", "_c"), validate="one_to_one")
mismatch = {}
for col in config.CLEAN_COLUMNS[1:]:
    a, b = m[col + "_t"], m[col + "_c"]
    if col == "Date":
        a, b = pd.to_datetime(a), pd.to_datetime(b)
    if pd.api.types.is_numeric_dtype(a):
        mismatch[col] = int(((a * 100).round() != (b * 100).round()).sum())
    else:
        mismatch[col] = int((a.fillna("").astype(str) != b.fillna("").astype(str)).sum())
print(f"True lines: {len(truth):,}   clean lines: {len(clean):,}   matched: {len(m):,}")
print(f"Cells that differ from the truth: {sum(mismatch.values())}")
missing = set(truth["Line_ID"]) - set(clean["Line_ID"])
print(f"True lines not in the clean file: {len(missing):,} (all of them are rows sent to review)")

True lines: 20,993   clean lines: 20,900   matched: 20,900
Cells that differ from the truth: 0
True lines not in the clean file: 93 (all of them are rows sent to review)


## 6. Validation checks written by the script

In [7]:
book["Validation_Checks"]

,Check,Result,Detail
0,Row reconciliation: raw = clean + blank + exact duplicates + merged copies + review,PASS,"21,815 = 20,900 + 25 + 525 + 252 + 113"
1,Line_ID is unique and well-formed,PASS,"20,900 unique of 20,900"
2,Target: Sales = Qty x Unit_Price on every row,PASS,0 rows differ
3,Target: Gross_Profit = Sales - Qty x Unit_Cost on every row,PASS,0 rows differ
4,Target: quantities are whole numbers >= 0,PASS,min 0
5,Target: unit price and unit cost > 0,PASS,NaN
6,Actual: Sales = Qty x Unit_Price on every row,PASS,0 rows differ
7,Actual: Gross_Profit = Sales - Qty x Unit_Cost on every row,PASS,0 rows differ
8,Actual: quantities are whole numbers >= 0,PASS,min 18
9,Actual: unit price and unit cost > 0,PASS,NaN


## 7. Large orders are flagged, not changed
Lines above 3x their target quantity pass every identity, so they are kept and flagged for a person to confirm.

In [8]:
flag = clean[clean["DQ_Flag"].fillna("") != ""]
print(f"Flagged lines: {len(flag):,} ({len(flag) / len(clean) * 100:.2f}% of lines)")
flag[["Line_ID", "Target_Qty", "Actual_Qty", "Actual_Sales_USD"]].head()

Flagged lines: 56 (0.27% of lines)


,Line_ID,Target_Qty,Actual_Qty,Actual_Sales_USD
217,EUR-HOM03-202401-007,346,1415,"99,007.55"
342,EUR-SPO04-202401-010,817,3875,"98,192.50"
700,NAM-SPO01-202401-014,150,803,"109,143.76"
726,NAM-SPO02-202401-006,347,1735,"52,882.80"
1056,APA-ELE06-202402-004,227,1130,"97,372.10"


**Re-running the cleaning** (from the repository root): `python scripts/clean_sales_data.py`